Implementazione Seq2Seq per Time Series

In [ ]:
import torch
import torch.nn as nn

class Seq2SeqForecaster(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, forecast_h, num_layers=2, dropout=0.2):
        super().__init__()
        self.forecast_h = forecast_h
        self.output_size = output_size

        self.encoder = nn.LSTM(
            input_size, hidden_size,
            num_layers, batch_first=True, dropout=dropout
        )

        self.decoder = nn.LSTM(
            output_size, hidden_size,
            num_layers, batch_first=True, dropout=dropout
        )

        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, src, trg=None, teacher_forcing_ratio=0.5):
        _, (h, c) = self.encoder(src)

        # Prende l'ultimo passo temporale della feature di output come input iniziale del decoder
        dec_input = src[:, -1:, :self.output_size]

        outputs = []
        for t in range(self.forecast_h):
            out, (h, c) = self.decoder(dec_input, (h, c))
            pred = self.linear(out)
            outputs.append(pred)

            # Teacher forcing
            if trg is not None and torch.rand(1).item() < teacher_forcing_ratio:
                dec_input = trg[:, t:t+1, :]
            else:
                dec_input = pred

        return torch.cat(outputs, dim=1)

Implementazione Modello Multimodale CNN+GRU per classificazione da dati visivi e serie temporale clinica

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class MultimodalClassifier(nn.Module):
    def __init__(self, img_feat_dim=256, ts_input_dim=20, ts_hidden=128, num_classes=2, dropout=0.5):
        super().__init__()

        # Backbone per le immagini
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.cnn = nn.Sequential(*list(backbone.children())[:-1])

        self.img_proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, img_feat_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Backbone per le serie storiche (Time Series)
        self.gru = nn.GRU(
            ts_input_dim, ts_hidden,
            num_layers=2, batch_first=True,
            dropout=dropout, bidirectional=True
        )

        # *2 perché la GRU è bidirezionale
        self.ts_proj = nn.Linear(ts_hidden * 2, img_feat_dim)

        # Classificatore finale
        self.classifier = nn.Sequential(
            nn.Linear(img_feat_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, img, ts, ts_lengths=None):
        # Estrazione feature immagini
        z_img = self.img_proj(self.cnn(img))

        # Estrazione feature serie storiche
        if ts_lengths is not None:
            ts = nn.utils.rnn.pack_padded_sequence(
                ts, ts_lengths.cpu(), batch_first=True, enforce_sorted=False
            )

        _, h_n = self.gru(ts)

        # Concatenazione degli ultimi stati forward e backward della GRU
        z_ts = self.ts_proj(torch.cat([h_n[-2], h_n[-1]], dim=-1))

        # Concatenazione feature multimodali e classificazione
        return self.classifier(torch.cat([z_img, z_ts], dim=-1))